In [1]:
import os
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd

#Change the path of your loca repository
sys_path = "Desktop/Proyectos/2026/02_Libreria_multimodal"
sys.path.insert(0, str(Path.home() / sys_path))
from multimodalpy.get_network import main

In [2]:
AREA_NAME = "València" #### CHANGE ###


CRS = "epsg:25830"
MODES = ["driving", "walking", "bike", "bus", "train"]
NAP_API_KEY= "44a2e8d8-fab3-4d87-940d-d7788a7dac94"

output_folder = Path.home() / "Desktop/descargas_multimodalpy/output_Valencia"
boundaries = "/Users/aidavillalba/Desktop/Proyectos/2026/02_Libreria_multimodal/multimodalpy/data/boundaries/layer_{}/layer_{}.shp".format(AREA_NAME, AREA_NAME)


In [3]:
from network_qa import (
    run_all_checks_osm,
    summary_matrix,
    style_summary_matrix,
    show_issues,
)

### Checks:

**Nodes**
* **nodes: expected columns present:** the nodes table contains all the expected columns (node_id, node_role, geometry).

* **no fully duplicate rows:** no row of nodes is identical to another in all its columns.
* **node_id is unique:** no node_id is duplicated.
* **node_id not null:** no node has an empty or NaN node_id.
* **geometry not null:** no node has an empty geometry.
* **node_role in [...]:** the value of node_role is one of the four valid values (endpoint, intersection, linear_vertex, through_endpoint).
* **geometry parses as valid POINT:** the geometry can be parsed and is a valid POINT (not a line, not corrupted).

**Edges**
* **edges: expected columns present:** the edges table contains all the columns expected by the OSM schema.

* **edge_id not null:** no edge has an empty edge_id.
* **from_node_id / to_node_id not null:** no edge has an empty source/destination node.
* **from_node_id / to_node_id exists in nodes table:** every node referenced by an edge actually exists in the nodes table (referential integrity).
* **no self-loop edges (from == to):** no edge starts and ends at the same node.
* **length not null / is non-negative:** the length column is present and contains no negative values.
* **tts (travel time) is non-negative:** the travel time is not negative.
* **oneway / reversed is boolean dtype:** these columns are of type True/False, not text (“true”/“1”/etc.).
* **geometry parses as valid LINESTRING:** the geometry of each edge is a valid line.
* **edge geometry endpoints match node coords:** the endpoints of the line match (within a tolerance) the coordinates of its from/to nodes.
* **length column matches geometry length:** the numerical value of ‘length’ matches (±2%) the actual calculated length of the geometry.
* **maxspeed numeric (...):** the speed limit is a number (or multiple values separated by ;, as is typical in OSM).
* **lanes numeric & positive (...):** the number of lanes is a positive number.
* **highway tag(s) within the recognised OSM vocabulary:** the value of the ‘highway’ tag is one of the recognised OSM tags (residential, primary, etc.), not an unusual or unexpected one.



In [4]:
STEM_BY_MODE = {
    "driving": "osm_driving",
    "walking": "osm_walking",
    "bike": "osm_bike",
    "train": "gtfs_20260902_080005_metro_valencia",  ##### Change creation date #####
    "bus": "gtfs_20260911_020033_genvalenciana_interurbano", ##### Change creation date #####
}


def load_network(mode: str, fmt: str, gpkg_path=None):

    stem = STEM_BY_MODE[mode]
    folder = Path(output_folder)

    if fmt == "geojson":
        nodes = gpd.read_file(folder / f"{stem}_nodes.geojson")
        edges = gpd.read_file(folder / f"{stem}_edges.geojson")
    elif fmt == "shapefile":
        nodes = gpd.read_file(folder / f"{stem}_nodes.shp")
        edges = gpd.read_file(folder / f"{stem}_edges.shp")
    elif fmt == "geopackage":
        if gpkg_path is None:
            raise ValueError("fmt='geopackage' requiere gpkg_path=...")
        gpkg_path = Path(gpkg_path)
        nodes = gpd.read_file(gpkg_path, layer=f"{stem}_nodes")
        edges = gpd.read_file(gpkg_path, layer=f"{stem}_edges")
    else:
        raise ValueError(f"Formato desconocido: {fmt!r}")

    return nodes, edges


def run_checks_for_all_modes(fmt: str, gpkg_path=None) -> dict:

    results_by_mode = {}
    for mode in MODES:
        nodes, edges = load_network(mode, fmt, gpkg_path=gpkg_path)
        results_by_mode[mode] = run_all_checks_osm(nodes, edges, verbose=False)
    return results_by_mode

def show_results(results_by_mode: dict):
    matrix = summary_matrix(results_by_mode)
    return style_summary_matrix(matrix)


## GeoJSON

In [5]:
main(
    area_name=AREA_NAME,
    modes=MODES,
    output_path=str(output_folder),
    crs=CRS,
    output_file_type="geojson",
    boundaries_path=str(boundaries),
    api_key=NAP_API_KEY,
)

[aviso] No se pudo normalizar el feed 20260727_080002_BUS_Samar.zip: 'stop_lat'
[aviso] No se pudo normalizar el feed 20260911_000015_BUS_Cantabria.zip: Python int too large to convert to C long
[aviso] No se pudo normalizar el feed 20260911_040014_RENFE_CERCA.zip: 'stop_sequence'
{
  "area_name": "València",
  "modes": [
    "driving",
    "walking",
    "bike",
    "bus",
    "train"
  ],
  "crs": "epsg:25830",
  "output_file_type": "geojson",
  "output_path": "/Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia",
  "osm_modes": [
    "driving",
    "walking",
    "bike"
  ],
  "gtfs_datasets": [
    "20260911_000046_bus_alsa",
    "20260911_020033_genvalenciana_interurbano",
    "20260910_200004_emt_valencia",
    "20260902_080005_metro_valencia",
    "20260702_060101_ouigo",
    "20260911_000009_renfe_avld"
  ],
  "files": [
    "study_area_boundary.geojson",
    "osm_driving_nodes.geojson",
    "osm_driving_edges.geojson",
    "osm_walking_nodes.geojson",
    "osm_w

{'area_name': 'València',
 'modes': ['driving', 'walking', 'bike', 'bus', 'train'],
 'crs': 'epsg:25830',
 'output_file_type': 'geojson',
 'output_path': '/Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia',
 'osm_modes': ['driving', 'walking', 'bike'],
 'gtfs_datasets': ['20260911_000046_bus_alsa',
  '20260911_020033_genvalenciana_interurbano',
  '20260910_200004_emt_valencia',
  '20260902_080005_metro_valencia',
  '20260702_060101_ouigo',
  '20260911_000009_renfe_avld'],
 'files': ['study_area_boundary.geojson',
  'osm_driving_nodes.geojson',
  'osm_driving_edges.geojson',
  'osm_walking_nodes.geojson',
  'osm_walking_edges.geojson',
  'osm_bike_nodes.geojson',
  'osm_bike_edges.geojson',
  'gtfs_20260911_000046_bus_alsa_nodes.geojson',
  'gtfs_20260911_000046_bus_alsa_edges.geojson',
  'gtfs_20260911_020033_genvalenciana_interurbano_nodes.geojson',
  'gtfs_20260911_020033_genvalenciana_interurbano_edges.geojson',
  'gtfs_20260910_200004_emt_valencia_nodes.geojson',
 

In [6]:
results_geojson = run_checks_for_all_modes("geojson")
show_results(results_geojson)

=== SUMMARY ===
26 passed, 1 failed, 0 warned (out of 27 checks)
=== SUMMARY ===
25 passed, 1 failed, 1 warned (out of 27 checks)
=== SUMMARY ===
25 passed, 1 failed, 1 warned (out of 27 checks)
=== SUMMARY ===
17 passed, 8 failed, 2 warned (out of 27 checks)
=== SUMMARY ===
17 passed, 8 failed, 2 warned (out of 27 checks)


,driving,walking,bike,bus,train
nodes: expected columns present,0,0,0,1,1
no fully duplicate rows,0,0,0,0,0
node_id is unique,0,0,0,0,0
node_id not null,0,0,0,0,0
geometry not null,0,0,0,0,0
"node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']",0,0,0,249,64
geometry parses as valid POINT,0,0,0,0,0
edges: expected columns present,0,0,0,8,8
edge_id not null,0,0,0,0,0
from_node_id not null,0,0,0,0,0


#### Analyse each check with errors:

In [7]:
show_issues(results_geojson["driving"], "maxspeed numeric (incl. ';'-separated values) when present").head(2)

,edge_id,from_node_id,to_node_id,highway,lanes,maxspeed,name,oneway,reversed,length,tts,geometry
1170,409287146,30378587,driving_topology_node_0000674,living_street,None,walk;20,Plaça de la Ciutat de Bruges;Plaça de la Comun...,False,True,7.642542,7.375658,"LINESTRING (725377.062 4372764.213, 725377.008..."
1171,4808259,30378587,driving_topology_node_0000674,living_street,None,walk;20,Plaça de la Ciutat de Bruges;Plaça de la Comun...,False,True,7.642542,7.375658,"LINESTRING (725377.062 4372764.213, 725377.008..."


In [8]:
show_issues(results_geojson["bus"], "node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']").head(2)

,node_id,stop_name,geometry
0,5252010122,"C. Sant Vicent Màrtir, 317",POINT (724939.125 4369674.708)
1,5252050117,Av. Fernando Abril Martorell - Hospital La Fe,POINT (725780.904 4369435.429)


## Shapefile

In [9]:
main(
    area_name=AREA_NAME,
    modes=MODES,
    output_path=str(output_folder),
    crs=CRS,
    output_file_type="shapefile",
    boundaries_path=str(boundaries),
    api_key=NAP_API_KEY,
)


[aviso] No se pudo normalizar el feed 20260727_080002_BUS_Samar.zip: 'stop_lat'
[aviso] No se pudo normalizar el feed 20260911_000015_BUS_Cantabria.zip: Python int too large to convert to C long
[aviso] No se pudo normalizar el feed 20260911_040014_RENFE_CERCA.zip: 'stop_sequence'


/Users/aidavillalba/Desktop/Proyectos/2026/02_Libreria_multimodal/multimodalpy/get_network.py:194: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(path, driver="ESRI Shapefile")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'matched_column' to 'matched_co'
  ogr_write(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'matched_value' to 'matched_va'
  ogr_write(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'matched_rows' to 'matched_ro'
  ogr_write(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'source_path' to 'source_

{
  "area_name": "València",
  "modes": [
    "driving",
    "walking",
    "bike",
    "bus",
    "train"
  ],
  "crs": "epsg:25830",
  "output_file_type": "shapefile",
  "output_path": "/Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia",
  "osm_modes": [
    "driving",
    "walking",
    "bike"
  ],
  "gtfs_datasets": [
    "20260911_000046_bus_alsa",
    "20260911_020033_genvalenciana_interurbano",
    "20260910_200004_emt_valencia",
    "20260902_080005_metro_valencia",
    "20260702_060101_ouigo",
    "20260911_000009_renfe_avld"
  ],
  "files": [
    "study_area_boundary.shp",
    "osm_driving_nodes.shp",
    "osm_driving_edges.shp",
    "osm_walking_nodes.shp",
    "osm_walking_edges.shp",
    "osm_bike_nodes.shp",
    "osm_bike_edges.shp",
    "gtfs_20260911_000046_bus_alsa_nodes.shp",
    "gtfs_20260911_000046_bus_alsa_edges.shp",
    "gtfs_20260911_020033_genvalenciana_interurbano_nodes.shp",
    "gtfs_20260911_020033_genvalenciana_interurbano_edges.shp",
   

{'area_name': 'València',
 'modes': ['driving', 'walking', 'bike', 'bus', 'train'],
 'crs': 'epsg:25830',
 'output_file_type': 'shapefile',
 'output_path': '/Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia',
 'osm_modes': ['driving', 'walking', 'bike'],
 'gtfs_datasets': ['20260911_000046_bus_alsa',
  '20260911_020033_genvalenciana_interurbano',
  '20260910_200004_emt_valencia',
  '20260902_080005_metro_valencia',
  '20260702_060101_ouigo',
  '20260911_000009_renfe_avld'],
 'files': ['study_area_boundary.shp',
  'osm_driving_nodes.shp',
  'osm_driving_edges.shp',
  'osm_walking_nodes.shp',
  'osm_walking_edges.shp',
  'osm_bike_nodes.shp',
  'osm_bike_edges.shp',
  'gtfs_20260911_000046_bus_alsa_nodes.shp',
  'gtfs_20260911_000046_bus_alsa_edges.shp',
  'gtfs_20260911_020033_genvalenciana_interurbano_nodes.shp',
  'gtfs_20260911_020033_genvalenciana_interurbano_edges.shp',
  'gtfs_20260910_200004_emt_valencia_nodes.shp',
  'gtfs_20260910_200004_emt_valencia_edges.shp'

In [10]:
results_shapefile = run_checks_for_all_modes("shapefile")
show_results(results_shapefile)

=== SUMMARY ===
22 passed, 4 failed, 0 warned (out of 26 checks)
=== SUMMARY ===
21 passed, 4 failed, 1 warned (out of 26 checks)
=== SUMMARY ===
21 passed, 4 failed, 1 warned (out of 26 checks)
=== SUMMARY ===
16 passed, 9 failed, 1 warned (out of 26 checks)
=== SUMMARY ===
16 passed, 9 failed, 1 warned (out of 26 checks)


,driving,walking,bike,bus,train
nodes: expected columns present,0,0,0,1,1
no fully duplicate rows,0,0,0,0,0
node_id is unique,0,0,0,0,0
node_id not null,0,0,0,0,0
geometry not null,0,0,0,0,0
"node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']",0,0,0,249,64
geometry parses as valid POINT,0,0,0,0,0
edges: expected columns present,1,1,1,9,9
edge_id not null,0,0,0,0,0
from_node_id not null,79686,397938,203748,219,1621


## GeoPackage

In [11]:
main(
    area_name=AREA_NAME,
    modes=MODES,
    output_path=str(output_folder),
    crs=CRS,
    output_file_type="geopackage",
    boundaries_path=str(boundaries),
    api_key=NAP_API_KEY,
)

[aviso] No se pudo normalizar el feed 20260727_080002_BUS_Samar.zip: 'stop_lat'
[aviso] No se pudo normalizar el feed 20260911_000015_BUS_Cantabria.zip: Python int too large to convert to C long
[aviso] No se pudo normalizar el feed 20260911_040014_RENFE_CERCA.zip: 'stop_sequence'
{
  "area_name": "València",
  "modes": [
    "driving",
    "walking",
    "bike",
    "bus",
    "train"
  ],
  "crs": "epsg:25830",
  "output_file_type": "geopackage",
  "output_path": "/Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia",
  "osm_modes": [
    "driving",
    "walking",
    "bike"
  ],
  "gtfs_datasets": [
    "20260911_000046_bus_alsa",
    "20260911_020033_genvalenciana_interurbano",
    "20260910_200004_emt_valencia",
    "20260902_080005_metro_valencia",
    "20260702_060101_ouigo",
    "20260911_000009_renfe_avld"
  ],
  "files": [
    "valencia.gpkg::study_area_boundary",
    "valencia.gpkg::osm_driving_nodes",
    "valencia.gpkg::osm_driving_edges",
    "valencia.gpkg:

{'area_name': 'València',
 'modes': ['driving', 'walking', 'bike', 'bus', 'train'],
 'crs': 'epsg:25830',
 'output_file_type': 'geopackage',
 'output_path': '/Users/aidavillalba/Desktop/descargas_multimodalpy/output_Valencia',
 'osm_modes': ['driving', 'walking', 'bike'],
 'gtfs_datasets': ['20260911_000046_bus_alsa',
  '20260911_020033_genvalenciana_interurbano',
  '20260910_200004_emt_valencia',
  '20260902_080005_metro_valencia',
  '20260702_060101_ouigo',
  '20260911_000009_renfe_avld'],
 'files': ['valencia.gpkg::study_area_boundary',
  'valencia.gpkg::osm_driving_nodes',
  'valencia.gpkg::osm_driving_edges',
  'valencia.gpkg::osm_walking_nodes',
  'valencia.gpkg::osm_walking_edges',
  'valencia.gpkg::osm_bike_nodes',
  'valencia.gpkg::osm_bike_edges',
  'valencia.gpkg::gtfs_20260911_000046_bus_alsa_nodes',
  'valencia.gpkg::gtfs_20260911_000046_bus_alsa_edges',
  'valencia.gpkg::gtfs_20260911_020033_genvalenciana_interurbano_nodes',
  'valencia.gpkg::gtfs_20260911_020033_genvalen

In [ ]:
gpkg_path = output_folder / "{}.gpkg".format("valencia") ### ".gpkg file without accents and lower case"
results_geopackage = run_checks_for_all_modes("geopackage", gpkg_path=gpkg_path)
show_results(results_geopackage)

=== SUMMARY ===
26 passed, 1 failed, 0 warned (out of 27 checks)
=== SUMMARY ===
25 passed, 1 failed, 1 warned (out of 27 checks)
=== SUMMARY ===
25 passed, 1 failed, 1 warned (out of 27 checks)
=== SUMMARY ===
17 passed, 8 failed, 2 warned (out of 27 checks)
=== SUMMARY ===
17 passed, 8 failed, 2 warned (out of 27 checks)


,driving,walking,bike,bus,train
nodes: expected columns present,0,0,0,1,1
no fully duplicate rows,0,0,0,0,0
node_id is unique,0,0,0,0,0
node_id not null,0,0,0,0,0
geometry not null,0,0,0,0,0
"node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']",0,0,0,249,64
geometry parses as valid POINT,0,0,0,0,0
edges: expected columns present,0,0,0,8,8
edge_id not null,0,0,0,0,0
from_node_id not null,0,0,0,0,0
